# Paper runs: surrogate scans and dedicated $N_b=2$ runs

Writes every BubbleMaster setup needed for the damped surrogate with a longer time window, plus one SLURM script per job:

1. **Dedicated N=2 runs** -- one row per lattice pair, at its exact $\gamma_{ij}$ (3 pairs at $\bar\lambda=0.84$, 3 at $\bar\lambda=0.069$), with each pair's $t_m$ inserted as an output time so the spectrum at $t_m$ is computed, not interpolated. wall_points 100; the time window ends at the current high-resolution scans' $T_{\rm max}$.
2. **N>3 scans** -- $\gamma_* = 1$--$6$ at $\bar\lambda=0.844777$ and $0.069$, wall_points 100, time_factor 2.5.
3. **$\gamma_*$-family scans** -- $\gamma_* = 1$--$16$ at $\bar\lambda=0.844777$ and $0.069$, wall_points 50, time_factor 2.5.

The time window of a scan is $T_{\rm max} = $ time_factor $\times R_*(\gamma_{*,\max})$. In a runtime scan the cutoff window is re-centred on every output time ($t_m = T$), so only its width $t_0 = 0.03\,d$ depends on the row.

**Workflow:** run the setup cells, fill in `SLURM` (nodes, walltime) for each job, run the SLURM cell, submit.

In [ ]:
from pathlib import Path

import h5py
import numpy as np
from scipy.optimize import brentq

import ptbuilder as pt
from ptbuilder.ic import BubbleMasterParams, generate_surrogate, write_2d_setup, _cutoff_times
from ptbuilder.patch_fraction import pair_separation
from ptbuilder.physics import Phi4Potential, PhysicsModel

ROOT = Path(pt.__file__).parent.parent

# ---------------------------------------------------------------- shared numerics (production values, as in 00_production_scan)
N_OMEGA, N_K   = 32, 51
N_MIN          = 128
PPW            = 40            # Filon panels per oscillation
HOW_OFTEN_DS   = 5
S_BATCH_SIZE   = 128
BABY_STEPS     = 100
CUTOFF_TYPE    = 0
LB845          = 0.8447772491349482

MODELS = {lb: PhysicsModel(Phi4Potential(lb)) for lb in (0.84, 0.069, LB845)}


def time_at_gamma(model, gamma):
    kin = model.kinematics
    return 0.0 if gamma <= 1.0 else brentq(lambda t: float(kin.Gamma(t)) - gamma, 0.0, 1.0e7)


def default_dt(model):
    """generate_surrogate's time step rule (scaled to the instanton size)."""
    return 2.0 * model.instanton.rmid_0 / 6.9364

## 1. Dedicated N=2 runs

$\gamma_{ij}$ are the exact values from the N=2 weights files (`data/weights_out_N3_084_pair_*_rnum.h5`, `data/weights_out_N3_069_pair_*_rnum.h5`). The setup's own separation $d$ (and so $t_m$) then matches the lattice pair's to $<10^{-4}$. Each row's frequencies span $kd$ = `KD_MIN`..`KD_MAX` in its own $d$.

In [ ]:
# ---------------------------------------------------------------- settings
N2_PAIRS = {                                   # lambda_bar -> exact pair gamma_ij
    0.84:  [4.006682957150458, 3.005961935699323, 2.006000568140851],
    0.069: [6.001495653377435, 5.0028178842027335, 4.001140112242929],
}
N2_T_END = {0.84: 104.0, 0.069: 18.81}         # = T_MAX of the current high-res scans (gs1-5 at 0.84, gs1-6_w100 at 0.069)
N2_WALL_POINTS = 100
KD_MIN, KD_MAX = 0.5, 150.0


# ---------------------------------------------------------------- setups
def write_n2_rows(lb):
    model = MODELS[lb]
    prof = model.instanton
    root = ROOT / 'data' / f'n2_dedicated_lb{lb:g}_w{N2_WALL_POINTS}'
    setup_dir, output_root = root / 'setups', root / 'gpu'
    if setup_dir.exists():
        raise FileExistsError(f'{root.relative_to(ROOT)} already exists -- remove it to rebuild')
    setup_dir.mkdir(parents=True); output_root.mkdir(parents=True)

    params = BubbleMasterParams(dz=None, wall_points=N2_WALL_POINTS, n_w=N_OMEGA, n_k=N_K, how_often_ds=HOW_OFTEN_DS,
                                baby_steps=BABY_STEPS, cutoff_type=CUTOFF_TYPE, t_0_scal=1.0, collision_radius='mid')
    dt, t_end = default_dt(model), N2_T_END[lb]
    rows = []
    for index, g in enumerate(N2_PAIRS[lb]):
        d = pair_separation(g, prof.rin_0, prof.rout_0, prof.rmid_0)
        t_m = _cutoff_times(d, CUTOFF_TYPE, 1.0)[2]
        t_first = time_at_gamma(model, g)
        times = np.linspace(t_first, t_end, int(np.ceil((t_end - t_first) / dt)) + 1)
        times = np.unique(np.r_[times, t_m])                    # t_m as an exact output time
        omega = np.geomspace(KD_MIN / d, KD_MAX / d, N_OMEGA)

        name = f'row_{index:04d}'
        setup = setup_dir / f'{name}.h5'
        write_2d_setup(model, g, times, setup, params, wlist=omega)
        with h5py.File(setup) as f:
            assert np.isclose(f.attrs['d'], d, rtol=1e-9), (f.attrs['d'], d)
        rows.append((index, g, len(times), setup.relative_to(ROOT), name))
        print(f'lb={lb:g}  gamma_ij={g:.6f}  d={d:.4f}  t_m={t_m:.4f}  t={t_first:.2f}..{t_end:.2f}  n_t={len(times)}')

    manifest = root / 'manifest.tsv'
    manifest.write_text('index\tgamma_ij\tn_t\tsetup\tname\n' + ''.join(f'{i}\t{g:.12g}\t{n}\t{s}\t{nm}\n' for i, g, n, s, nm in rows))
    return dict(tag=f'n2_dedicated_lb{lb:g}_w{N2_WALL_POINTS}', manifest=manifest, output_root=output_root, n_rows=len(rows))


JOBS = {}
for lb in N2_PAIRS:
    job = write_n2_rows(lb)
    JOBS[job['tag']] = job

## 2.--3. Scans

Built with `generate_surrogate`. The scan folder name does not encode wall_points or time_factor, so both go into the tag (e.g. `runtime_scan_lb0.844777_w100_tf2.5_gs1-6_kR1-100_nw32`) -- this also keeps them from landing in an existing scan's folder.

In [ ]:
# ---------------------------------------------------------------- settings
SCAN_DEFS = [   # (lambda_bar, gamma_* min, gamma_* max, wall_points, time_factor)
    (LB845, 1.0, 6.0,  100, 2.5),
    (0.069, 1.0, 6.0,  100, 2.5),
    (LB845, 1.0, 16.0, 50,  2.5),
    (0.069, 1.0, 16.0, 50,  2.5),
]
KR_MIN, KR_MAX = 1.0, 100.0


# ---------------------------------------------------------------- setups
for lb, gs_min, gs_max, wp, tf in SCAN_DEFS:
    label = f'lb{lb:g}_w{wp}_tf{tf:g}'
    tag = f'{label}_gs{gs_min:g}-{gs_max:g}_kR{KR_MIN:g}-{KR_MAX:g}_nw{N_OMEGA}'
    if (ROOT / 'data' / f'runtime_scan_{tag}').exists():
        raise FileExistsError(f'data/runtime_scan_{tag} already exists -- remove it to rebuild')
    scan = generate_surrogate(MODELS[lb], ROOT, label, gs_min, gs_max, KR_MIN, KR_MAX, N_OMEGA,
                              wall_points=wp, time_factor=tf, n_k=N_K, n_min=N_MIN, panels_per_oscillation=PPW,
                              how_often_ds=HOW_OFTEN_DS, s_batch_size=S_BATCH_SIZE, cutoff_type=CUTOFF_TYPE,
                              baby_steps=BABY_STEPS)
    assert scan.tag == tag, (scan.tag, tag)
    JOBS[tag] = dict(tag=tag, manifest=scan.manifest, output_root=scan.output_root, n_rows=len(scan.manifest_rows))
    print(f'{tag}: {len(scan.manifest_rows)} rows, gamma_ij {scan.gamma_ij_grid[0]:.2f}-{scan.gamma_ij_grid[-1]:.2f}, '
          f'T_MAX={scan.t_max:.2f} (= {tf:g} R_*({gs_max:g}))')

## SLURM scripts

One script per job, rows dispatched largest-$\gamma_{ij}$-first (see the sizing discussion in 00_production_scan). Fill in nodes and walltime per job; jobs left at `None` are skipped. Size them from a short test run.

In [ ]:
# ---------------------------------------------------------------- settings
GPU_ACCOUNT = 'YOUR_ACCOUNT'   # SLURM account to charge
GPU_QOS     = 'premium'
SLURM = {   # tag -> (nodes, walltime)
    'n2_dedicated_lb0.84_w100':                         (None, None),
    'n2_dedicated_lb0.069_w100':                        (None, None),
    'lb0.844777_w100_tf2.5_gs1-6_kR1-100_nw32':         (None, None),
    'lb0.069_w100_tf2.5_gs1-6_kR1-100_nw32':            (None, None),
    'lb0.844777_w50_tf2.5_gs1-16_kR1-100_nw32':         (None, None),
    'lb0.069_w50_tf2.5_gs1-16_kR1-100_nw32':            (None, None),
}


# ---------------------------------------------------------------- scripts
def write_slurm_script(job, gpu_nodes, gpu_walltime):
    tag = job['tag']
    gpus_per_node = 4
    n_concurrent = gpu_nodes * gpus_per_node
    manifest = job['manifest'].relative_to(ROOT)
    output = job['output_root'].relative_to(ROOT)
    (ROOT / 'logs').mkdir(exist_ok=True)
    script_path = ROOT / 'scripts' / f'{tag}_gpu.sh'
    script_path.write_text(f'''#!/bin/bash
#SBATCH --job-name=bm_{tag}_gpu
#SBATCH --output={ROOT}/logs/bm_{tag}_gpu_%j.out
#SBATCH --constraint=gpu
#SBATCH --qos={GPU_QOS}
#SBATCH -A {GPU_ACCOUNT}
#SBATCH --nodes={gpu_nodes}
#SBATCH --ntasks={n_concurrent}
#SBATCH --ntasks-per-node={gpus_per_node}
#SBATCH --cpus-per-task=32
#SBATCH --gpus-per-task=1
#SBATCH --time={gpu_walltime}
set -euo pipefail
REPO_ROOT="{ROOT}"
cd "$REPO_ROOT"

run_one() {{
    local row_num=$1
    local line
    line=$(sed -n "$((row_num + 1))p" {manifest})
    IFS=$'\\t' read -r index gamma n_t setup name <<< "$line"
    output="{output}/${{name}}"
    mkdir -p "$output"
    /usr/bin/time --verbose srun --exclusive --exact -N 1 -n 1 -c 32 --gpus-per-task=1 --cpu-bind=cores bin/bubblemaster_gpu "$setup" "$output/" --param {N_MIN} --filon-panels-per-osc {PPW} --s-batch-size {S_BATCH_SIZE}
}}
export -f run_one

seq {job['n_rows']} -1 1 | xargs -P {n_concurrent} -I ROW bash -c 'run_one "$@"' _ ROW
''')
    script_path.chmod(0o755)
    return script_path


for tag, job in JOBS.items():
    nodes, walltime = SLURM[tag]
    if nodes is None or walltime is None:
        print(f'{tag}: nodes/walltime not set -- skipped')
        continue
    path = write_slurm_script(job, nodes, walltime)
    print(f'{tag}: {job["n_rows"]} rows on {nodes * 4} GPUs  ->  sbatch {path.relative_to(ROOT)}')